In [10]:
from explore_import import  *
import ionbot_preprocess as io
import data_preprocess as dt
import hpp_checker as hpp
from Download_UnimodDB import *
import tpp_preprocess as tpp

from pyteomics import mass
import itertools 
from itertools import combinations
import time
import re
import ast
from scipy.spatial import distance
from gql import gql, Client
from gql.transport.aiohttp import AIOHTTPTransport
import asyncio
import plotly.graph_objects as go
warnings.simplefilter(action='ignore', category=FutureWarning)

In [5]:
# GLOBAL VARS
PIPELINE = 'ionbot_open'
PIPELINE_LW = PIPELINE.lower()

In [6]:
#base directories

root="/project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery"
processed_dir=f"{root}/oui-discovery-vv-data/processed/20251015-oui-discovery-ionbot-results"
# get processed files
processed_paths=dt.list_files(processed_dir)
# leave only non-entrapment and open search, only search files
tmp = processed_paths.copy()
for parent, files in processed_paths.items():
    if '-closed' in parent or '-entrap' in parent:
        del tmp[parent]
        continue
    if not any( 'group-walk-output.csv' in f for f in files):
        del tmp[parent]
processed_paths = tmp

# where to save pickles
results_dir = f"{root}/oui-discovery-vv-data/pickles"

20251015-oui-discovery-ionbot-results/
    PXD005833.v0.11.4/
        openprot-x-trembl-filt-global-outerjoin.csv
        PXD005833-opensearch-x-closedsearch-filt-global.csv.gz
        Metadata_PXD005833.v0.11.4-openprot.txt
        ID-rate-df-filt-hybrid.csv
        ID-rate-df-filt-False.csv
        Metadata_PXD005833.v0.11.4-canon.txt
        Metadata_PXD005833.v0.11.4-trembl.txt
        PXD005833-opensearch-x-closedsearch-filt-global-outerjoin.csv.gz
        openprot-x-trembl-filt-global.csv
        ID-rate-df-filt-custom.csv
        ID-rate-df-filt-global.csv
        ID-rate-df-filt-groupwalk.csv
        PXD005833.v0.11.4-canon/
            PXD005833.v0.11.4-canon-combined-features.csv.gz
            combined-results-w-qvalues.csv.gz
            AM15-canon/
                ionbot.first.proteins.csv
                sample-protein-inference-input.pout
                group-walk-output-peptide-vv.csv
                ionbot.features.csv
                ionbot.first.csv
                

In [7]:
def get_peptide_position(database_peptide, protein, OPfasta):
    count_var=0
    #print(database_peptide, protein, count_var)
    prot_seq=str(OPfasta[protein].seq)
    pos=hpp.get_pep_positions(prot_seq, database_peptide)
    if len(pos)>1: pos=[pos[0]] #take first occurance in protein
    if len(pos)==0: #if didnt find peptide in protein
        # try L->I
        database_peptide = database_peptide.replace('L','I')
        prot_seq=str(OPfasta[protein].seq)
        pos=hpp.get_pep_positions(prot_seq, database_peptide)
        if len(pos)>1: pos=[pos[0]]
        if len(pos)==0:
            count_var+=1;  pos=[np.nan]
    if count_var > 0: print(database_peptide, protein, count_var)
    return pos[0]

In [8]:
OPfasta_file="/project/def-marie87/vvshazia/pride_reanalysis/CompOmics/oui-discovery/oui-discovery-vv-data/raw/msfragger_pride_reanalysis/concatenated-fastas/20241102-openprot-database_ionbot_db/concat.fasta"
input_file = open(OPfasta_file)
OPfasta = SeqIO.to_dict(SeqIO.parse(OPfasta_file, "fasta"))

#### Load PSM

In [11]:
psm_filt = []
for parent, files in processed_paths.items():
    if ".ipynb_checkpoints" in parent: continue
    for file in files:
        if file == 'group-walk-output.csv': #-db
            spectrum_file = '-'.join(parent.split('/')[-1].split('-')[:-1])
            search_database = parent.split('/')[-1].split('-')[-1]
            dataset = parent.split('/')[-2].split('-')[0]
            data = pd.read_csv(os.path.join(parent, file))
            data['spectrum_file'] = spectrum_file
            data['search_database'] = search_database
            data['dataset'] = dataset
            # from Enrico
            # fixes issue with some files being .RAW and other being .raw
            #data.spectrum_file = data.spectrum_file.apply(lambda x: x.split('.')[0])
            # in some mgf files the 'spectrum title' includes the file name, making the spectrum title unique.
            # when the file name is NOT included, spectrum titles are NOT unique, and this can mess up some analysis.
            #data.spectrum_title = data.spectrum_file + ':' + data.spectrum_title.apply(lambda x: x.split(':')[-1])
            psm_filt += [data]
psm_filt = pd.concat(psm_filt)
# NOTE: decoys are retained
psm_filt = psm_filt[(psm_filt['q.value'] < 0.01)&(psm_filt['isCanonical'] != 'Contam')]

# str to list
psm_filt['proteins'] = psm_filt['proteins'].apply(lambda x: x.split(';'))
# indicate proteotypic peptides
psm_filt['isProteotypic'] = psm_filt['proteins'].apply(lambda x: len(x)==1)
# peptide class
psm_filt["peptide_class"]=psm_filt["proteins"].apply(tpp.classify_peptide_tpp)

/tmp/ipykernel_3905008/2595102683.py:9: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(os.path.join(parent, file))
/tmp/ipykernel_3905008/2595102683.py:9: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(os.path.join(parent, file))


#### Load peptide

In [12]:
pep_filt = []
for parent, files in processed_paths.items():
    if ".ipynb_checkpoints" in parent: continue
    for file in files:
        if file == 'group-walk-output-peptide-vv.csv' :
            spectrum_file = '-'.join(parent.split('/')[-1].split('-')[:-1])
            search_database = parent.split('/')[-1].split('-')[-1]
            dataset = parent.split('/')[-2].split('-')[0]
            data = pd.read_csv(os.path.join(parent, file))
            data['spectrum_file'] = spectrum_file
            data['search_database'] = search_database
            data['dataset'] = dataset
            pep_filt += [data]
pep_filt = pd.concat(pep_filt)
# NOTE: only target peptides are left
pep_filt = pep_filt[(pep_filt['subset_max_rank'] == True)&(pep_filt['isCanonical'] != 'Contam')]

# str to list
#pep_filt['protein'] = pep_filt['protein'].apply(lambda x: ast.literal_eval(x))
pep_filt['proteins'] = pep_filt['proteins'].apply(lambda x: x.split(';'))
# indicate proteotypic peptides
pep_filt['isProteotypic'] = pep_filt['proteins'].apply(lambda x: len(x)==1)
# peptide class
pep_filt["peptide_class"]=pep_filt["proteins"].apply(tpp.classify_peptide_tpp)

#### Merge PSM and peptide

In [13]:
# Select PSM maching peptide level 
psmpep_filt_op = psm_filt[psm_filt['search_database'] == 'openprot']
print('All OP psm', len(psmpep_filt_op))
psmpep_filt_op = psmpep_filt_op[psmpep_filt_op['database_peptide'].isin(pep_filt[pep_filt['search_database'] == 'openprot']['database_peptide'].tolist())]
psmpep_filt_op.reset_index(drop=True,inplace=True)
print('Peptide-level psm', len(psmpep_filt_op))
# Select proteotypic non-canonical peptides-psm
psmpep_prnc=psmpep_filt_op[psmpep_filt_op.peptide_class=="unique_to_Noncanon"]
print('Proteotypic non-canon peptide-level psm',len(psmpep_prnc))

All OP psm 323529
Peptide-level psm 313941
Proteotypic non-canon peptide-level psm 4568


In [15]:
del psmpep_filt_op

In [16]:
#add peptide length and parse peptide position
psmpep_prnc["peptide_length"]=psmpep_prnc.database_peptide.apply(lambda x: len(x))

# indicate peptide position
psmpep_prnc["peptide_position"]='nan'
psmpep_prnc['peptide_position'] = psmpep_prnc.apply(lambda x: get_peptide_position(x['database_peptide'], x['proteins'][0], OPfasta), axis=1)

#### Annotate modifications

In [17]:
def PTM_string_to_masses_list(ptm_string, unimod):
    if ptm_string=='Unmodified':
        return 'Unmodified'
    # if a peptide has 2 or more possible modified forms, take only 1
    x = ptm_string.split('_or_')[0]
    x = x.split('||')
    x = [_.split('|') for _ in x]
    x = {a:re.split('[][]',b) for a,b in x}
    x = [(int(a),b[3],int(b[1])) for a,b in x.items()]
    x = [(a,b,getPTMmass(c,unimod)) for a,b,c in x]    
    return x
#Modify to parce 9999 not by unimod_id but by code_name
def PTM_string_to_masses_list_9999(ptm_string, unimod, unimod_df):
    if ptm_string=='Unmodified':
        return 'Unmodified'
    # if a peptide has 2 or more possible modified forms, take only 1
    #print(ptm_string)
    x = ptm_string.split('_or_')[0]
    x = x.split('||')
    x = [_.split('|') for _ in x]
    #print(x)
    x = {a:re.split('[][]',b) for a,b in x}
    #print(x)
    x = [(int(a),b[3],int(b[1]) if "9999" not in b[1] else b[2]) for a,b in x.items()]
    x = [(a,b,getPTMmass(c,unimod) if not isinstance(c,str) 
          else float(unimod_df[unimod_df.code_name==c].mono_mass.iloc[0]) if c in unimod_df.code_name.tolist()
         else np.nan) for a,b,c in x]    
    return x

In [18]:
def Download_Unimod_Dict_names():
    unimod = Download_UnimodDB()
    condensed_unimod = {}
    for uni_id,df in unimod.groupby("unimod_id").__iter__():
        condensed_unimod[uni_id] = {}
        condensed_unimod[uni_id]['residues'] = "".join([_ for _ in df.residue if '-' not in _])
        condensed_unimod[uni_id]['mono_mass'] = [_ for _ in set(df.mono_mass)][0]
        condensed_unimod[uni_id]['code_name'] = [_ for _ in set(df.code_name)][0]
        condensed_unimod[uni_id]['full_name'] = [_ for _ in set(df.full_name)][0]
        condensed_unimod[uni_id]['classification'] = [_ for _ in set(df.classification)][0]
        
    return condensed_unimod

In [20]:
# 3-letter to 1-letter amino acid mapping
aa_map = {
    "Ala": "A", "Cys": "C", "Asp": "D", "Glu": "E", "Phe": "F",
    "Gly": "G", "His": "H", "Ile": "I", "Lys": "K", "Leu": "L",
    "Met": "M", "Asn": "N", "Pro": "P", "Gln": "Q", "Arg": "R",
    "Ser": "S", "Thr": "T", "Val": "V", "Trp": "W", "Tyr": "Y",
    "Xle": "L",  # Xle usually refers to Leu/Ile, mapped to Leu ("L") here
    "CamCys": "C", "MetOx": "M"  # Optional: include PTMs if needed
}

# Your list of mutations
mutations = ['His2Asn', 'His2Asp', 'Ala->Ser', 'Ala->Thr', 'Ala->Asp', ...]  # etc.

# Function to extract 1-letter code of the target amino acid
def extract_target_aa(mutation):
    if "->" in mutation:
        target = mutation.split("->")[1]
    elif "2" in mutation:
        target = mutation.split("2")[1]
    else:
        return None  # or raise error if unexpected format

    return aa_map.get(target, "?")  # return '?' if unknown

In [21]:
unimod = Download_Unimod_Dict_names()
unimod_df=pd.DataFrame.from_dict(unimod, orient='index')
#mark substitutions
unimod_df["isSubstitution"]=unimod_df.full_name.apply(lambda x: "substitution" in x)
unimod_df_subs=unimod_df[unimod_df["isSubstitution"]]
#to what residue is change
unimod_df_subs["subs_to_residue"]=unimod_df_subs.code_name.apply(lambda x: extract_target_aa(x))

/tmp/ipykernel_3905008/210214903.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  unimod_df_subs["subs_to_residue"]=unimod_df_subs.code_name.apply(lambda x: extract_target_aa(x))


In [23]:
# Remove retention times in parentheses
psmpep_prnc['modifications_noRT'] = psmpep_prnc['modifications'].str.replace(r'\([^)]*\)', '', regex=True)

# Convert modifications to mass lists
psmpep_prnc['modifications_masses'] = psmpep_prnc['modifications'].map(
    lambda x: PTM_string_to_masses_list_9999(x, unimod, unimod_df)
)

# Extract list of masses and total mass shift in one pass
def extract_mass_info(mods):
    if mods == 'Unmodified':
        return [0], 0
    masses = [mod[2] for mod in mods]
    return masses, np.sum(masses)

mass_info = psmpep_prnc['modifications_masses'].map(extract_mass_info)
psmpep_prnc[['masses', 'tot_mass_shift']] = pd.DataFrame(mass_info.tolist(), index=psmpep_prnc.index)
psmpep_prnc['tot_mass_shift'] = psmpep_prnc['tot_mass_shift'].fillna(0)

# Precompute lookups for substitutions and classification
mass_to_residues = unimod_df_subs.groupby('mono_mass')['residues'].apply(set).to_dict()
mass_to_class = unimod_df.groupby('mono_mass')['classification'].apply(list).to_dict()
known_sub_masses = set(mass_to_residues.keys())

def has_substitution(peptide, masses):
    for m in masses:
        if m in known_sub_masses:
            residues = mass_to_residues[m]
            if any(aa in residues for aa in peptide):
                return True
    return False

psmpep_prnc['isSubstitution'] = [
    has_substitution(peptide, masses)
    for peptide, masses in zip(psmpep_prnc['database_peptide'], psmpep_prnc['masses'])
]

psmpep_prnc['masses_classification'] = psmpep_prnc['masses'].map(
    lambda masses: [mass_to_class.get(m, []) for m in masses]
)

### Check non-canonical proteins identifications against HPP guidlines

In [31]:
def filter_nested_intervals(intervals):
    if len(intervals)==1: return True
    # Sort by start ascending, and end descending to prioritize wider intervals
    #unpack the list
    #intervals=[i[0] for i in intervals]
    sorted_intervals = sorted(intervals, key=lambda x: (x[0], -x[1]))
    result = []
    for i, (start_i, end_i) in enumerate(sorted_intervals):
        is_nested = False
        for j, (start_j, end_j) in enumerate(sorted_intervals):
            if i != j and start_j <= start_i and end_i <= end_j:
                is_nested = True
                break
        if not is_nested:
            result.append((start_i, end_i))
    result_bool=[i in result for i in intervals]
    return result_bool
def filter_short_overlaps(i,overlapping_intervals,minimal_length):
    indexes,intervals = overlapping_intervals
    for indx,interval in zip(indexes,intervals):
        if i in indx and len(indx)==1:
            return True
        elif i in indx and len(indx)>1:
            return abs(interval[0]-interval[1])>=minimal_length
        else:
            continue

In [37]:
def track_hpp_countdown(countdown_dict, newkey, pepdf, psmdf):
    countdown_dict["psm"][newkey]=len(psmdf[psmdf.database_peptide.isin(pepdf.database_peptide.tolist())])
    countdown_dict["pep"][newkey]=len(pepdf)
    countdown_dict["prot"][newkey]=len(pepdf.leadprot.unique())
    return countdown_dict
    
def is_goodlength(df,min_length=9):
    '''Filteres out peptides with length shorter than minimal.'''
    return df[df.peptide_length>=min_length]

def is_nonnested(df):
    '''Filteres out sub-peptides. Apply only on proteotypic peptides.'''
    df["NonNested"] = df.groupby("leadprot")["peptide_position"].transform(
        lambda intervals: filter_nested_intervals(intervals.values)
    )
    return df[df["NonNested"]]

def mark_overlaps(group,minimal_length=18):
    intervals = group.peptide_position.values
    if len(intervals) == 1:
        group["overlap_intervals"] = [True] * len(group)
    else:
        #unpack the list
        #intervals=[i[0] for i in intervals]
        overlapping_intervals = hpp.find_max_length_in_overlap_groups(intervals)
        good_overlap = [
            filter_short_overlaps(i, overlapping_intervals, minimal_length) 
            for i in range(len(group))
        ]
        group["overlap_intervals"] = good_overlap
    return group

def is_nonoverlap(df):
    df = df.groupby("leadprot", group_keys=False).apply(mark_overlaps)
    return df[df.overlap_intervals]
    
def check_2_pep(group):
    return len(group)>=2

def is_2pep(df):
    df["isProt2Pep"]=df.leadprot.apply(lambda x: check_2_pep(df.groupby("leadprot").get_group(x)))
    return df[df.isProt2Pep]

In [38]:
#track how many we loose with each HPP criteria
hpp_countdown={"psm":{},"pep":{},"prot":{}}

In [39]:
#track
hpp_countdown["psm"]["original"]=len(psmpep_prnc)
hpp_countdown["pep"]["original"]=len(psmpep_prnc.drop_duplicates("database_peptide"))
hpp_countdown["prot"]["original"]=len(psmpep_prnc.leadprot.unique())

#### Peptide level

In [40]:
psmpep_prnc_pepfilt=psmpep_prnc.copy(deep=True)

#Go to peptide level - leave 1 PSM per peptide
psmpep_prnc_pepfilt.drop_duplicates("database_peptide",inplace=True)

#Filter out peptides <9 aa long
psmpep_prnc_pepfilt=is_goodlength(psmpep_prnc_pepfilt)

#track <9aa
hpp_countdown=track_hpp_countdown(hpp_countdown, "<9aa", psmpep_prnc_pepfilt, psmpep_prnc)

#Filter out nested peptides
psmpep_prnc_pepfilt=is_nonnested(psmpep_prnc_pepfilt)

#track nested
hpp_countdown=track_hpp_countdown(hpp_countdown, "nested", psmpep_prnc_pepfilt, psmpep_prnc)

#For overlapping peptides, check the total extent length is >=18
psmpep_prnc_pepfilt=is_nonoverlap(psmpep_prnc_pepfilt)

#track <18aa
hpp_countdown=track_hpp_countdown(hpp_countdown, "<18aa", psmpep_prnc_pepfilt, psmpep_prnc)

/tmp/ipykernel_3905008/2451256502.py:34: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("leadprot", group_keys=False).apply(mark_overlaps)


In [44]:
#Filter out peptides, that are non-unqiely maped to peptide variants

#Input to the list into https://hppportal.net/toolchecker.html
#for pep in psmpep_prnc_pepfilt.database_peptide:
#    print(pep)
psmpep_prnc_pepfilt["database_peptide"].to_csv(f"{results_dir}/nc_uniqness_check.csv", index=False, header=False)

In [53]:
#read the output
uniqness_checker=pd.read_csv(f"{results_dir}/nc_uniqnesschecker_res.csv")
uniqness_checker["NoMatch"]=uniqness_checker['Subject Accession']=='NO MATCH'
#mark uniqness
psmpep_prnc_pepfilt=psmpep_prnc_pepfilt.merge(uniqness_checker[["Peptide","NoMatch"]],left_on="database_peptide",right_on="Peptide",how="left")
psmpep_prnc_pepfilt=psmpep_prnc_pepfilt[psmpep_prnc_pepfilt.NoMatch!=False]

#track uniqness
hpp_countdown=track_hpp_countdown(hpp_countdown, "uniqness", psmpep_prnc_pepfilt, psmpep_prnc)

#check for unique 2 peptides, filterout if not
psmpep_prnc_pepfilt=is_2pep(psmpep_prnc_pepfilt)

#track min 2 proteotypic peptides per protein
hpp_countdown=track_hpp_countdown(hpp_countdown, "2pep1", psmpep_prnc_pepfilt, psmpep_prnc)

#### PSM level

##### Proteotypic peptides with isoSAAV explained by gnomAD

##### Proteotypic peptides with isoSAAV explained by other peptide in database

In [ ]:
#check that substitution location is on expected residue

comb_datasets_ncun_pepfilt_psmisosub["isSubAA"]=np.nan
comb_datasets_ncun_pepfilt_psmisosub["isSubMass"]=np.nan
for i, row in comb_datasets_ncun_pepfilt_psmisosub[comb_datasets_ncun_pepfilt_psmisosub.isSubstitution].iterrows():
    modifications_masses=row.modifications_masses
    for pos, aa, mass in modifications_masses:
        info=unimod_df_subs[unimod_df_subs.mono_mass==mass]
        if len(info[info.residues.str.contains(aa)]):
            comb_datasets_ncun_pepfilt_psmisosub.loc[i,["isSubAA","isSubMass"]]=[aa,mass]
print("number of peptides with expected PTM-residue pair",comb_datasets_ncun_pepfilt_psmisosub[comb_datasets_ncun_pepfilt_psmisosub.isSubstitution].isSubAA.isna().value_counts())

In [ ]:
#however, we are not sure if location is correct, so get mass and compare it to shared peptides of same mass with 1-aa distance
# mass is already available in column peptide_mass (includes mass shift)

mass_error=0.000100
for i, row in comb_datasets_ncun_pepfilt_psmisosub[comb_datasets_ncun_pepfilt_psmisosub.isSubstitution].iterrows():
    peptide_length=len(row.database_peptide)
    peptide_mass=row.peptide_mass
    candidates=insituPEP.loc[((insituPEP.peptide!=row.database_peptide)&(insituPEP.peptide_length==peptide_length)&(abs(insituPEP.peptide_mass-peptide_mass)<=mass_error))]
    #dedupl
    candidates.drop_duplicates("peptide",inplace=True)
    #check for 1 aa distance
    ham=1/peptide_length
    candidates["1_aa_dist"]=candidates.peptide.apply(lambda x: distance.hamming(list(x),list(row.database_peptide))<=ham)
    comb_datasets_ncun_pepfilt_psmisosub.loc[i,["1_aa_dist"]]="|".join(candidates[candidates["1_aa_dist"]].peptide.tolist()) if len(candidates[candidates["1_aa_dist"]])>0 else False

In [ ]:
#how many non-canonical proteorypic peptides with isoSAAV can be explained by canonical peptide in database
comb_datasets_ncun_pepfilt_psmisosub.loc[comb_datasets_ncun_pepfilt_psmisosub.isSubstitution][["1_aa_dist"]].value_counts()

In [ ]:
# how many peptides had non-expected residue location with substitution, but found SAAV canonical peptide
comb_datasets_ncun_pepfilt_psmisosub[(comb_datasets_ncun_pepfilt_psmisosub.isSubstitution)&(
    comb_datasets_ncun_pepfilt_psmisosub["1_aa_dist"]!=False)&(comb_datasets_ncun_pepfilt_psmisosub.isSubAA.isna())][["database_peptide","modifications_masses","1_aa_dist"]]
#acetylation, with substitutions in S->T ; G->A ; V->I ; but different location
#how do we treat this instanses??? 
#If location is predicted diferent, but peptide mass is very similar, then it is a question of location prediction quality aka spectra qc

##### Merge peptide and psm level filtering results

### Compile the table for spectra visualisation of all HPP-pass peptides

### Visualise filtering steps

### How many proteins out of detected could be theoreticaly pass HPP?